In [76]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup as B
import time

# 設定 Selenium 瀏覽器
driver = webdriver.Chrome()  # 確保您已安裝 ChromeDriver

title_urls = []
title_list = []

for i in range(1,10):
    kkbox_url = f'https://www.kkbox.com/tw/tc/search/lyrics?q=TRASH&p={i}'

    # 使用 Selenium 打開網頁
    driver.get(kkbox_url)
    time.sleep(3)  # 等待 JavaScript 加載內容

    # 取得頁面內容
    html = driver.page_source
    soup = B(html, 'html.parser')

    for song_item in soup.find_all('song-li'):
        title_a = song_item.find('a', slot='song-name')  # 精確定位
        if title_a:
            song_url = title_a['href']
            song_title = title_a['title']
            title_urls.append(song_url)
            title_list.append(song_title)

# 關閉瀏覽器，將driver.quit()放迴圈外，確保在所有頁面爬取完成後再關閉瀏覽器 
driver.quit()

import pandas as pd
df = pd.DataFrame({'Title': title_list, 'url':title_urls})

In [82]:
import requests as req
from bs4 import BeautifulSoup as B
import time
lyrics_list = []

for url in df['url']:
    resp = req.get(url)

    if resp.status_code == 200:
        soup = B(resp.text, 'html.parser')
        lyrics_div = soup.find('div', class_="lyrics")
        lyrics_p = lyrics_div.find_all('p')
        lyrics = lyrics_p[1].get_text(separator=" ", strip = True)

    lyrics_list.append(lyrics)

    time.sleep(3)

import pandas as pd
df['lyrics'] = lyrics_list
df = df.drop(columns=['url'])

In [1]:
import jieba
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 載入資料
lda_data = pd.read_csv('TRASH_kkbox_cleaned.csv')

# 分詞處理
lda_data['Processed_Lyrics'] = lda_data['Lyrics'].apply(lambda x: ' '.join(jieba.cut(str(x))))

# TF-IDF 向量化
stop_words = ['the', 'and', 'it', 'my', 'in', 'to', 'la', 'da', 'of', 'na', 'll', 'don']
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2, stop_words=stop_words)
tfidf_matrix = tfidf_vectorizer.fit_transform(lda_data['Processed_Lyrics'])

# LDA 模型
lda_tfidf_model = LatentDirichletAllocation(n_components=3, random_state=42)
lda_tfidf_model.fit(tfidf_matrix)

# 抓取主題關鍵詞
def get_top_words(model, feature_names, n_top_words):
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        topics[f"Topic {topic_idx + 1}"] = top_words
    return topics

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_topics = get_top_words(lda_tfidf_model, tfidf_feature_names, 10)

# 顯示結果
topics_df = pd.DataFrame.from_dict(tfidf_topics, orient='index', columns=[f"Top Word {i+1}" for i in range(10)])
topics_df


Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\USER\AppData\Local\Temp\jieba.cache
Loading model cost 0.602 seconds.
Prefix dict has been built successfully.


,Top Word 1,Top Word 2,Top Word 3,Top Word 4,Top Word 5,Top Word 6,Top Word 7,Top Word 8,Top Word 9,Top Word 10
Topic 1,這個,什麼,sucker,眼前,無所謂,回來,goodbye,一切,我們,一直
Topic 2,全部,這樣,所有,結果,穿過,還有,傷口,singing,怎麼,忘記
Topic 3,you,me,that,we,yeah,up,no,all,your,love


In [7]:
import jieba
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# 載入資料
lda_data = pd.read_csv('cocktail_ingredients.csv')

# 分詞處理
lda_data['Processed_Ingredients'] = lda_data['Ingredients'].apply(lambda x: ' '.join(jieba.cut(str(x))))

# TF-IDF 向量化
tfidf_vectorizer = TfidfVectorizer(max_df=0.95, min_df=2)
tfidf_matrix = tfidf_vectorizer.fit_transform(lda_data['Processed_Ingredients'])

# LDA 模型
lda_tfidf_model = LatentDirichletAllocation(n_components=5, random_state=42)
lda_tfidf_model.fit(tfidf_matrix)

# 抓取主題關鍵詞
def get_top_words(model, feature_names, n_top_words):
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]]
        topics[f"Topic {topic_idx + 1}"] = top_words
    return topics

tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()
tfidf_topics = get_top_words(lda_tfidf_model, tfidf_feature_names, 10)

# 顯示結果
topics_df = pd.DataFrame.from_dict(tfidf_topics, orient='index', columns=[f"Top Word {i+1}" for i in range(10)])
topics_df


,Top Word 1,Top Word 2,Top Word 3,Top Word 4,Top Word 5,Top Word 6,Top Word 7,Top Word 8,Top Word 9,Top Word 10
Topic 1,rum,pineapple,juice,jamaica,lime,syrup,coffee,de,coconut,liqueur
Topic 2,juice,rum,syrup,lime,wine,water,light,sherry,simple,red
Topic 3,juice,lime,syrup,lemon,liqueur,simple,orange,mint,bitters,gin
Topic 4,bitters,lemon,angostura,juice,whiskey,syrup,egg,white,rye,grenadine
Topic 5,gin,vermouth,dry,twist,orange,grapefruit,lemon,juice,campari,tequila


In [6]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Step 1: Load the data
file_path = 'cocktail_ingredients.csv'
cocktail_data = pd.read_csv(file_path)

# Step 2: Preprocess the data
# Clean and lower the ingredient text
cocktail_data['Cleaned_Ingredients'] = cocktail_data['Ingredients'].str.lower().str.replace(r'[\s]+', ' ', regex=True)

# Remove rows with missing or NaN values in the 'Cleaned_Ingredients' column
cocktail_data = cocktail_data.dropna(subset=['Cleaned_Ingredients'])

# Step 3: Vectorize the ingredient text
vectorizer = CountVectorizer(token_pattern=r'\b\w+\b', stop_words='english')
ingredient_matrix = vectorizer.fit_transform(cocktail_data['Cleaned_Ingredients'])

# Step 4: Fit the LDA model
lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(ingredient_matrix)

# Step 5: Define a function to extract topics
def display_topics(model, feature_names, num_top_words=10):
    topics = {}
    for topic_idx, topic in enumerate(model.components_):
        topics[f"Topic {topic_idx + 1}"] = [feature_names[i] for i in topic.argsort()[:-num_top_words - 1:-1]]
    return topics

# Step 6: Extract and display topics
feature_names = vectorizer.get_feature_names_out()
topics = display_topics(lda, feature_names)

# Output topics
for topic, keywords in topics.items():
    print(f"{topic}: {', '.join(keywords)}")

Topic 1: lemon, juice, bitters, syrup, angostura, orange, whiskey, bourbon, simple, cinnamon
Topic 2: rum, juice, lime, pineapple, syrup, liqueur, jamaica, light, cream, coconut
Topic 3: orange, bitters, vermouth, twist, sweet, liqueur, dry, gin, whiskey, cherry
Topic 4: mint, syrup, sprig, juice, lime, leaves, lemon, simple, chartreuse, liqueur
Topic 5: juice, syrup, lime, lemon, simple, gin, water, soda, orange, grapefruit


In [101]:
import pyLDAvis
import numpy as np
from sklearn.preprocessing import normalize

# Enable notebook mode (if you're in Jupyter Notebook)
pyLDAvis.enable_notebook()

# Prepare LDA model results for visualization
def prepare_lda_vis(lda_model, doc_term_matrix, vectorizer):
    topic_term_dists = normalize(lda_model.components_, norm='l1', axis=1)
    doc_topic_dists = normalize(lda_model.transform(doc_term_matrix), norm='l1', axis=1)
    doc_lengths = np.array(doc_term_matrix.sum(axis=1)).flatten()
    vocab = vectorizer.get_feature_names_out()
    term_frequency = np.array(doc_term_matrix.sum(axis=0)).flatten()

    # Use pyLDAvis prepare function
    vis_data = pyLDAvis.prepare(
    topic_term_dists=topic_term_dists,
    doc_topic_dists=doc_topic_dists,
    doc_lengths=doc_lengths,
    vocab=vocab,
    term_frequency=term_frequency,
    sort_topics=False  # 保留 LDA 模型中的原始主題順序
    )
    return vis_data

# Call the function
lda_vis_data = prepare_lda_vis(lda_model, doc_term_matrix, vectorizer)

# Display visualization
pyLDAvis.display(lda_vis_data)
